# Bronze Ingestion: Fingrid Electricity Data

Fetches raw consumption and production data from the Fingrid Open Data API 
and writes it to Bronze Delta tables without transformation.

**Datasets:** consumption (124), production (74)<br>
**Source:** https://data.fingrid.fi/api<br>
**Output:** bronze.fingrid_consumption, bronze.fingrid_production<br>

In [0]:
import requests
import time

from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

Creates the `electricity_project` catalog and its `bronze` schema so all tables for this project are grouped under one catalog.

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS electricity_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS electricity_project.bronze")

DataFrame[]

Retrieves the Fingrid API key from Databricks Secrets rather than hardcoding 
it, so the credential never appears in the notebook source or version control.

In [0]:
api_key = dbutils.secrets.get(scope="electricity-project", key="fingrid-api-key")

Fetches all pages of a dataset from the Fingrid API, handling pagination 
and respecting the API's rate limit (1 request per 2 seconds) with a delay 
between page requests.

In [0]:
def fetch_dataset(dataset_id, start_time, end_time, api_key):
    all_rows = []
    page = 1
    while True:
        url = f"https://data.fingrid.fi/api/datasets/{dataset_id}/data"
        params = {
            "startTime": start_time,
            "endTime": end_time,
            "pageSize": 20000,
            "page": page
        }
        headers = {"x-api-key": api_key}
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        result = response.json()
        all_rows.extend(result["data"])
        next_page = result["pagination"].get("nextPage")
        if next_page is None:
            break
        page = next_page
        time.sleep(2.5)
    return all_rows

Fetches one year of consumption (dataset 124) and 
production (dataset 74) data, then prints the row counts as a sanity check.

In [0]:
start_time = "2025-09-01T00:00:00Z"
end_time = "2026-09-01T00:00:00Z"

consumption_data = fetch_dataset(124, start_time, end_time, api_key)
production_data = fetch_dataset(74, start_time, end_time, api_key)

print(len(consumption_data), len(production_data))

35037 35039


Converts the fetched data into Spark DataFrames and writes them as Delta 
tables in the bronze schema, preserving the raw structure from the API.

In [0]:
schema = StructType([
    StructField("datasetId", LongType(), True),
    StructField("startTime", StringType(), True),
    StructField("endTime", StringType(), True),
    StructField("value", DoubleType(), True)
])

consumption_df = spark.createDataFrame(consumption_data, schema=schema)
production_df = spark.createDataFrame(production_data, schema=schema)

consumption_df.write.mode("overwrite").saveAsTable("electricity_project.bronze.fingrid_consumption")
production_df.write.mode("overwrite").saveAsTable("electricity_project.bronze.fingrid_production")

Sanity check.

In [0]:
display(spark.table("electricity_project.bronze.fingrid_consumption").limit(5))
display(spark.table("electricity_project.bronze.fingrid_production").limit(5))

datasetId,startTime,endTime,value
124,2025-12-01T06:30:00.000Z,2025-12-01T06:45:00.000Z,11585.3
124,2025-12-01T06:15:00.000Z,2025-12-01T06:30:00.000Z,11541.7
124,2025-12-01T06:00:00.000Z,2025-12-01T06:15:00.000Z,11436.3
124,2025-12-01T05:45:00.000Z,2025-12-01T06:00:00.000Z,11371.6
124,2025-12-01T05:30:00.000Z,2025-12-01T05:45:00.000Z,11358.6


datasetId,startTime,endTime,value
74,2025-10-16T14:45:00.000Z,2025-10-16T15:00:00.000Z,11108.3
74,2025-10-16T14:30:00.000Z,2025-10-16T14:45:00.000Z,10889.8
74,2025-10-16T14:15:00.000Z,2025-10-16T14:30:00.000Z,10861.5
74,2025-10-16T14:00:00.000Z,2025-10-16T14:15:00.000Z,10875.8
74,2025-10-16T13:45:00.000Z,2025-10-16T14:00:00.000Z,10988.2
